# Legal Reducto OracleDB Demo Notebook

This notebook demonstrates the legal version of the Reducto + Oracle flow:

1. Load `.env`.
2. Connect to Oracle and ensure the schema exists.
3. Inspect documents tagged with `filing_type = 'LEGAL'`.
4. Retrieve chunk evidence for holdings, judgments, parties, dates, and obligations.
5. Optionally ingest a legal document URL through Reducto.

Legal retrieval is chunk-first here. The optional ingest cell stores chunks and deliberately clears parsed tables and financial facts.

## Platform Context

![Executive thesis: Reducto AI + Oracle Database](assets/reducto_oracle_executive_thesis.svg)

![Capability map: Oracle Database approach vs standalone vector platforms](assets/reducto_oracle_platform_capability_map.svg)

![Operating model and risk controls](assets/reducto_oracle_operating_model_risk_controls.svg)


In [1]:
import os
import sys
from pathlib import Path
from dataclasses import replace


def find_sdk_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "src" / "reducto" / "lib" / "oracledb").exists():
            return path
    raise RuntimeError("Could not locate the reducto-python-sdk repository root")


ROOT = find_sdk_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_env(path=ROOT / "examples" / "oracledb" / ".env"):
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[len("export ") :].strip()
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip("'\"")


def optional_int(name):
    value = os.getenv(name)
    return int(value) if value else None


load_env()

DOMAIN_LABEL = "LEGAL"
DOMAIN_ENTITY = os.getenv("LEGAL_ENTITY", "DEMO_LEGAL")
DOMAIN_YEAR = optional_int("LEGAL_YEAR")
QUESTION = os.getenv(
    "LEGAL_DEMO_QUESTION",
    "What did the court decide, hold, or order?",
)

print("Oracle user:", os.getenv("ORACLE_USER"))
print("Oracle DSN:", os.getenv("ORACLE_DSN"))
print("Reducto key set:", bool(os.getenv("REDUCTO_API_KEY")))
print("Cohere key set:", bool(os.getenv("CO_API_KEY")))
print("Domain label:", DOMAIN_LABEL)

Oracle user: REDUCTO_RAG_APP
Oracle DSN: localhost:1521/FREEPDB1
Reducto key set: True
Cohere key set: True
Domain label: LEGAL


## Connect to Oracle and inspect the schema

In [2]:
from reducto.lib.oracledb.config import connect_oracle, vector_dimensions_from_env
from reducto.lib.oracledb.oracle import OracleSchemaManager

connection = connect_oracle()
OracleSchemaManager(connection).create_schema(vector_dimensions=vector_dimensions_from_env())
print("Oracle schema is ready")

for table in ["DOCUMENTS", "DOCUMENT_CHUNKS", "PARSED_TABLES", "FINANCIAL_FACTS"]:
    with connection.cursor() as cursor:
        cursor.execute(f"select count(*) from {table}")
        print(table, cursor.fetchone()[0])

Oracle schema is ready
DOCUMENTS 3
DOCUMENT_CHUNKS 28
PARSED_TABLES 52
FINANCIAL_FACTS 1222


## Ingest a legal URL

Set `RUN_LEGAL_INGEST=1` and `LEGAL_SOURCE_URI=<url>` before running this cell. The notebook stores chunks only by clearing parsed tables and financial facts before persistence.

In [ ]:
RUN_INGEST = os.getenv("RUN_LEGAL_INGEST") == "1"
SOURCE_URI = os.getenv("LEGAL_SOURCE_URI", "")

if not RUN_INGEST:
    print("Skipping ingest. Set RUN_LEGAL_INGEST=1 and LEGAL_SOURCE_URI to parse a document.")
elif not SOURCE_URI:
    raise RuntimeError("LEGAL_SOURCE_URI must be set when RUN_LEGAL_INGEST=1")
else:
    from reducto.lib.oracledb.models import DocumentMetadata
    from reducto.lib.oracledb.oracle import OracleDocumentRepository
    from reducto.lib.oracledb.embeddings import embedding_provider_from_env
    from reducto.lib.oracledb.reducto_client import ReductoDocumentParser

    parse_result = ReductoDocumentParser().parse_url(SOURCE_URI)
    parse_result = replace(parse_result, tables=[], financial_facts=[])
    metadata = DocumentMetadata(
        source_uri=SOURCE_URI,
        source_kind="url",
        company=DOMAIN_ENTITY,
        fiscal_year=DOMAIN_YEAR,
        filing_type=DOMAIN_LABEL,
        title=os.getenv("LEGAL_TITLE", "Legal Reducto demo document"),
    )
    document_id = OracleDocumentRepository(connection).store_parse_result(
        metadata,
        parse_result,
        embedding_provider_from_env(input_type="search_document"),
    )
    print("Stored legal document", document_id)
    print("Chunks:", len(parse_result.chunks), "Tables stored:", len(parse_result.tables))

## List stored legal documents

In [5]:
def fetch_domain_documents(limit=10):
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            select document_id, company, fiscal_year, filing_type, title, source_uri
            from documents
            where upper(filing_type) = upper(:domain_label)
            order by created_at desc
            fetch first {max(1, int(limit))} rows only
            """,
            domain_label=DOMAIN_LABEL,
        )
        return cursor.fetchall()


domain_docs = fetch_domain_documents()
if not domain_docs:
    print("No LEGAL documents found yet. Use the optional ingest cell below to add one.")
else:
    for row in domain_docs:
        print(row)

(42, 'SCOTUS', 2024, 'LEGAL', 'Trump v. Anderson Supreme Court Opinion', 'https://www.supremecourt.gov/opinions/23pdf/23-719_19m2.pdf')


## Retrieve legal chunk evidence

This cell uses a lightweight SQL keyword search so the notebook can run without a live embedding request. Legal does not need parsed tables for the core demo; chunks are enough for evidence-backed answers.

In [6]:
from reducto.lib.oracledb.qa import format_answer, answer_from_search_results
from reducto.lib.oracledb.utils import read_lob
from reducto.lib.oracledb.models import SearchResult

SEARCH_TERMS = ["court", "held", "hold", "judgment", "affirmed", "reversed", "ordered"]


def lexical_chunk_search(terms, limit=5):
    clauses = []
    params = {"domain_label": DOMAIN_LABEL}
    for index, term in enumerate(terms):
        key = f"term_{index}"
        clauses.append(f"lower(c.content) like :{key}")
        params[key] = f"%{term.lower()}%"
    term_sql = " or ".join(clauses)
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            select d.document_id, c.chunk_id, c.content, d.company, d.fiscal_year,
                   d.filing_type, c.page_start, c.page_end, d.source_uri
            from document_chunks c
            join documents d on d.document_id = c.document_id
            where upper(d.filing_type) = upper(:domain_label)
              and ({term_sql})
            order by c.created_at desc
            fetch first {max(1, int(limit))} rows only
            """,
            params,
        )
        rows = cursor.fetchall()

    results = []
    for row in rows:
        content = str(read_lob(row[2]) or "")
        score = sum(1 for term in terms if term.lower() in content.lower())
        results.append(
            SearchResult(
                document_id=int(row[0]),
                chunk_id=int(row[1]),
                score=float(score),
                content=content,
                company=row[3],
                fiscal_year=int(row[4]) if row[4] is not None else None,
                filing_type=row[5],
                page_start=int(row[6]) if row[6] is not None else None,
                page_end=int(row[7]) if row[7] is not None else None,
                source_uri=str(row[8] or ""),
            )
        )
    return results


if not domain_docs:
    print("Skipping evidence retrieval because no LEGAL documents are stored yet.")
else:
    results = lexical_chunk_search(SEARCH_TERMS, limit=5)
    if not results:
        print("No keyword matches found for:", SEARCH_TERMS)
    else:
        print(format_answer(answer_from_search_results(QUESTION, results, evidence_limit=3)))

Question
  What did the court decide or hold?

Answer
  It reaches out to decide Section 3 questions not before us, and to foreclose future efforts to disqualify a Presidential candidate under that provision.

Evidence
  1. page 19
     See ante, at 5 (quoting 11 F. Cas., at 26). Once again, even petitioner's lawyer distanced himself from fully embracing this case as probative of Section 3's meaning. See Tr. of Oral Arg. 35–36. The majority also cites Senator Trumbull's statements that Section 3 "'provide[d] no means for enforcing'" itself. Ante, at 5 (quoting Cong. Globe, 41st Cong., 1st Sess., 626 (1869)). The majority, however, neglects to mention the Senator's view that "[i]t is the [F]ourteenth [A]mendment that prevents a person from holding office," with the proposed legislation simply "affor[ding] a more efficient and speedy remedy" for effecting the disqualification. Cong. Globe, 41st Cong., 1st Sess., at 626–627. Ultimately, under the guise of providing a more "complete explan

## Optional semantic search

This calls the configured embedding provider, so it is opt-in.

In [7]:
RUN_LIVE_EMBED_SEARCH = os.getenv("RUN_LIVE_EMBED_SEARCH") == "1"

if not RUN_LIVE_EMBED_SEARCH:
    print("Skipping live semantic search. Set RUN_LIVE_EMBED_SEARCH=1 to enable it.")
elif not domain_docs:
    print("Skipping semantic search because no LEGAL documents are stored yet.")
else:
    from reducto.lib.oracledb.models import SearchFilters
    from reducto.lib.oracledb.retrieval import OracleHybridRetriever
    from reducto.lib.oracledb.embeddings import embedding_provider_from_env

    retriever = OracleHybridRetriever(
        connection,
        embedding_provider_from_env(input_type="search_query"),
    )
    semantic_results = retriever.semantic_search(
        QUESTION,
        filters=SearchFilters(filing_type=DOMAIN_LABEL),
        limit=5,
    )
    print(format_answer(answer_from_search_results(QUESTION, semantic_results, evidence_limit=3)))

Question
  What did the court decide or hold?

Answer
  In this case, the Court must decide whether Colorado may keep a Presidential candidate off the ballot on the ground that he is an oathbreaking insurrectionist and thus disqualified from holding fe

Evidence
  1. page 15
     D STATES No. 23–719# DONALD J. TRUMP, PETITIONER v. NORMA ANDERSON, ET AL. ON WRIT OF CERTIORARI TO THE SUPREME COURT OF COLORADO [March 4, 2024] JUSTICE SOTOMAYOR , JUSTICE KAGAN , and JUSTICE JACKSON , concurring in the judgment. "If it is not necessary to decide more to dispose of a case, then it is necessary not to decide more." Dobbs v. Jackson Women's Health Organization, 597 U. S. 215, 348 (2022) (ROBERTS, C. J., concurring in judgment). That fundamental principle of judicial restraint is practically as old as our Republic. This Court is authorized "to say what the law is" only because "[t]hose who apply [a] rule to particular cases must of necessity expound and interpret that rule." Marbury v. Madison,

In [7]:
connection.close()
print("Connection closed")

Connection closed
